# GRPO benchmark: the zebra task with meepsquared and Unsloth

<small>Copyright Alejandro Martínez Corriá and the Thinkube contributors. SPDX-License-Identifier: Apache-2.0</small>

This notebook runs a few steps of GRPO on the zebra task and compares the arms on the time of a step and the share of it spent generating:

| Arm | What it is |
|---|---|
| `meepsquared` | meepsquared, the model generating with Transformers |
| `meepsquared-vllm` | meepsquared, the model generating through vLLM in the training process (TRL's `GRPOConfig(use_vllm=True, vllm_mode="colocate")`) |
| `unsloth` | Unsloth, generating with Transformers |

The zebra task: the model translates each clue of a zebra puzzle into a rule, a constraint solver deduces the grid, and the reward is the share of correct cells. Qwen3.5-4B, LoRA rank 16 on the seven projections, 4 prompts of 8 completions of up to 800 tokens per step, the recipe's sampling and optimizer. Each run is one call of `grpo/grpo.py` and appends one line to `results/grpo.jsonl`: the first step, the mean of the later steps, the time generating in each step, the rewards.

## Settings

`MODEL_SOURCE` says where the model comes from: `"huggingface"` downloads it from the Hub; `"thinkube"` reads it from the cluster's model mirror on [Thinkube](https://thinkube.org). `UNSLOTH_PYTHON` is the Python of an environment with Unsloth; the `meepsquared-vllm` arm needs vLLM in this kernel's environment, and is skipped with a note when it is not there.

In [ ]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "thinkube" or "huggingface"
MODEL = PARAMETERS.get("MODEL", "Qwen/Qwen3.5-4B")
ARMS = PARAMETERS.get("ARMS", ["meepsquared", "meepsquared-vllm", "unsloth"])
STEPS = PARAMETERS.get("STEPS", 4)
# Environment variables for the runs, by arm, for example {"unsloth": {"UNSLOTH_COMPILE_DISABLE": "1"}}
ENV = PARAMETERS.get("ENV", {"unsloth": {"UNSLOTH_COMPILE_DISABLE": "1"}})
# The Python of an environment with Unsloth; on Thinkube, the custom venv "unsloth"
UNSLOTH_PYTHON = PARAMETERS.get("UNSLOTH_PYTHON")
# The benchmarks folder of meepsquared: this notebook's folder, unless a job runs a copy elsewhere
BENCHMARKS = PARAMETERS.get("BENCHMARKS")

In [ ]:
import importlib.util
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

BENCHMARKS = Path(BENCHMARKS) if BENCHMARKS else Path.cwd()
if not (BENCHMARKS / "grpo" / "grpo.py").is_file():
    raise FileNotFoundError(f"grpo/grpo.py is not in {BENCHMARKS}: run this notebook from the benchmarks folder of meepsquared, "
                            "or set BENCHMARKS to that folder")
RESULTS = BENCHMARKS / "results" / "grpo.jsonl"

if UNSLOTH_PYTHON is None and "unsloth" in ARMS:
    arch = {"aarch64": "arm64", "x86_64": "amd64"}[platform.machine()]
    UNSLOTH_PYTHON = f"/home/thinkube/venvs/custom/unsloth/{arch}/bin/python"
    if not Path(UNSLOTH_PYTHON).is_file():
        raise FileNotFoundError(f"{UNSLOTH_PYTHON} does not exist: build the venv 'unsloth', "
                                "or set UNSLOTH_PYTHON to the Python of an environment with Unsloth")
VLLM = importlib.util.find_spec("vllm") is not None

sys.path.insert(0, str(BENCHMARKS))
from common import environment

print(json.dumps(environment(), indent=1))
print("vLLM in this environment:", VLLM)

## Where the model comes from

On Thinkube, `thinkube_models.mirrored_model_path` gives the folder of the mirrored model, and stops with an error naming it if it is not mirrored, before any training starts.

In [ ]:
if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    print(f"{MODEL}: {tkm.mirrored_model_path(MODEL)}")
else:
    print("The model is downloaded from the Hugging Face Hub")

## Run

Each run prints its steps as it goes. A run that fails stops the notebook with its own error.

In [ ]:
def run(arm):
    script = [str(BENCHMARKS / "grpo" / "grpo.py"), "--arm", arm.split("-")[0], "--model", MODEL, "--source", MODEL_SOURCE,
              "--steps", str(STEPS), "--results", str(RESULTS)]
    if arm == "meepsquared-vllm":
        command = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", "1", *script, "--vllm"]
    else:
        command = [UNSLOTH_PYTHON if arm == "unsloth" else sys.executable, "-u", *script]
    print(f"=== {MODEL} {arm}", flush=True)
    process = subprocess.Popen(command, cwd=BENCHMARKS / "grpo", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, "TOKENIZERS_PARALLELISM": "false", **ENV.get(arm, {})})
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"{MODEL} {arm} failed with exit code {process.returncode}")


for arm in ARMS:
    if arm == "meepsquared-vllm" and not VLLM:
        print("=== meepsquared-vllm: skipped, vLLM is not in this environment", flush=True)
        continue
    run(arm)

## Results

The first step includes the kernels' compilation; the later steps are the measure. The generation time is most of a GRPO step: 32 completions of up to 800 tokens.

In [ ]:
import pandas as pd

rows = [json.loads(line) for line in RESULTS.read_text().splitlines()]
table = pd.DataFrame([r for r in rows if r["model"] == MODEL and r.get("source") == MODEL_SOURCE and r["steps"] == STEPS])
table = table.drop_duplicates(["arm"], keep="last").set_index("arm")
table["generation_s_later_steps"] = table["generation_s_by_step"].map(lambda g: round(sum(v for k, v in g.items() if int(k) > 1) / max(1, len(g) - 1), 1))
display(table.loc[[a for a in ARMS if a in table.index], ["first_step_s", "s_per_step_after_first", "generation_s_later_steps", "peak_gib", "rewards"]])